# NB_03 -- Gold: Star Schema

Builds the Gold layer -- the final BI-ready dataset consumed by Power BI.

Gold organizes data into a star schema with two fact tables sharing conformed dimensions
(Constellation Schema pattern):
- fact_sales: one row per order line (17,032 rows)
- fact_budget: one row per employee per year (after unpivot)

All dimensions are fully denormalized. Power BI needs no joins at the report layer.

Metrics computed in fact_sales:
- SalesAmount: quantity x historical unit price x (1 - discount)
- TotalProductSales: same as SalesAmount -- pure product revenue label
- TotalOrderValue: SalesAmount + freight (total order value including shipping)
- GrossMargin: SalesAmount minus cost of goods sold
- GrossMarginPct: margin as a percentage of SalesAmount
- DeliveryDays: shipment date minus order date
  Note: this will be negative because shipment data is from a legacy system (2007-2012)
  predating all orders (2016-2021). Flagged in ShipmentDataWarning column.


In [ ]:

from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd

print("Gold layer -- Star Schema build started")


In [ ]:
# gold_dim_date
# Generated programmatically to cover both the order range (2016-2021)
# and the shipment range (2007-2012) so all date keys resolve in the model.
# Includes fiscal year attributes: fiscal year starts in April per the scenario
# (last sale March 30, 2021 falls in FY2020, not FY2021).

dates = pd.date_range("2007-01-01", "2021-12-31", freq="D")
pdf_dates = pd.DataFrame({
    "DateKey":      [int(d.strftime("%Y%m%d")) for d in dates],
    "FullDate":     dates,
    "Year":         dates.year,
    "Quarter":      dates.quarter,
    "Month":        dates.month,
    "MonthName":    dates.strftime("%B"),
    "Week":         dates.isocalendar().week.astype(int),
    "Day":          dates.day,
    "DayOfWeek":    dates.dayofweek + 1,
    "DayName":      dates.strftime("%A"),
    "YearMonth":    dates.strftime("%Y-%m"),
    "FiscalYear":   [d.year if d.month >= 4 else d.year - 1 for d in dates],
    "FiscalQuarter":[
        "FQ1" if d.month in (4,5,6) else
        "FQ2" if d.month in (7,8,9) else
        "FQ3" if d.month in (10,11,12) else
        "FQ4" for d in dates
    ],
    "Season":       [
        "Spring" if d.month in (3,4,5) else
        "Summer" if d.month in (6,7,8) else
        "Fall"   if d.month in (9,10,11) else
        "Winter" for d in dates
    ],
})

df_date = spark.createDataFrame(pdf_dates)
df_date.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("gold_dim_date")
print(f"gold_dim_date: {df_date.count()} rows | {df_date.select('Year').distinct().count()} years")


In [ ]:
# gold_dim_customer
# Joins silver_customers with silver_divisions to bring DivisionName.
# Continent, SubRegion, and ISO2 are already in silver_customers (enriched in NB_02).
# DivisionID was derived from Country in Silver, so the data here is clean.

df_cust = spark.table("silver_customers")
df_div  = spark.table("silver_divisions")

df_dim = (df_cust
    .join(df_div, "DivisionID", "left")
    .select(
        "CustomerID",
        col("CompanyName").alias("CustomerName"),
        "ContactName", "City", "StateProvince", "Country", "ISO2",
        "DivisionID", "DivisionName", "Continent", "SubRegion",
    )
)

df_dim.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("gold_dim_customer")
print(f"gold_dim_customer: {df_dim.count()} rows")
df_dim.groupBy("DivisionName","Continent").count().orderBy("DivisionName").show()


In [ ]:
# gold_dim_product
# Joins products with categories and suppliers.
# MarginBand, PriceTier, and InventoryStatus are already computed in Silver.

from pyspark.sql.functions import col, coalesce, current_timestamp
df_prod = spark.table("silver_products")
df_cat  = spark.table("silver_categories")
df_sup  = spark.table("silver_suppliers")

df_dim = (df_prod
    .join(df_cat, "CategoryID", "left")
    .join(df_sup.select("SupplierID",
                         col("CompanyName").alias("SupplierName"),
                         col("Country").alias("SupplierCountry"),
                         col("Continent").alias("SupplierContinent")),
          "SupplierID", "left")
    .select(
        "ProductID", "ProductName",
        "CategoryID", "CategoryName",
        "SupplierID", "SupplierName", "SupplierCountry", "SupplierContinent",
        "UnitCost", "UnitPrice",
        "MarginAbs", "MarginPct", "MarginBand",
        "PriceTier", "InventoryStatus",
    )
)

df_dim.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("gold_dim_product")
print(f"gold_dim_product: {df_dim.count()} rows")


In [ ]:
# gold_dim_employee
# Joins employees with offices to bring office location details.
# The Unknown Employee record (ID = -1) ensures that the 5 orders with no assigned
# sales rep still appear in all employee-level reports without breaking aggregations.

from pyspark.sql.functions import col, concat_ws, coalesce, current_timestamp, lit

df_emp = spark.table("silver_employees")
df_off = spark.table("silver_offices").select(
    col("Office").alias("OfficeKey"),
    "OfficeCity", "OfficeCountry", "OfficeStateProvince",
    col("Continent").alias("OfficeContinent"),
    col("SubRegion").alias("OfficeSubRegion")
)

df_dim = (df_emp
    .join(df_off, df_emp["Office"] == df_off["OfficeKey"], "left")
    .drop("OfficeKey")
    .select(
        "EmployeeID",
        concat_ws(" ", col("First_Name"), col("Last_Name")).alias("FullName"),
        "First_Name", "Last_Name", "Title",
        "Office", "OfficeCity", "OfficeCountry", "OfficeStateProvince",
        "OfficeContinent", "OfficeSubRegion",
        "Year_Salary", "Salary_Confidential", "Hire_Date",
    )
)

df_dim.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("gold_dim_employee")
print(f"gold_dim_employee: {df_dim.count()} rows")
df_dim.show(5)


In [ ]:
# gold_dim_shipper
# Includes Unknown Shippers 4 and 5 that were injected in NB_02.
# Without them, roughly 20% of orders would have no shipper dimension.

df_dim = spark.table("silver_shippers").select(
    "ShipperID",
    col("CompanyName").alias("ShipperName")
)
df_dim.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("gold_dim_shipper")
print(f"gold_dim_shipper: {df_dim.count()} rows")
df_dim.show()


In [ ]:
# gold_fact_sales
# Central fact Table
# Line-level fact. Grain: one row per order line.
# Freight excluded intentionally -- freight is an order-level cost, not a line cost.
# Summing freight at line grain would inflate totals by N lines per order.
# Freight lives in gold_fact_orders where grain matches the cost.

from pyspark.sql.functions import date_format
from pyspark.sql.functions import (
    col, coalesce, lit, when,
    round as spark_round, sum as spark_sum
)

df_od   = spark.table("silver_order_details")
df_ord  = spark.table("silver_orders").select(
    "OrderID","CustomerID","EmployeeID","OrderDate",
    "FiscalYear","OrderYear","OrderMonth","OrderQuarter","OrderSeason"
)
df_prod = spark.table("silver_products").select("ProductID","UnitCost")

# v2 NOTE: GrossMargin uses current UnitCost from Products, not historical cost at time of sale.
# This is a known limitation: if UnitCost changed over 2016-2021, historical margins are inaccurate.
# GrossMarginPct is a ROW-LEVEL metric. Do NOT average it in Power BI.
# For correct aggregate margin, use DAX: DIVIDE(SUM(GrossMargin), SUM(SalesAmount))
#
# Line-level fact: no freight, no order-level attributes
df_items = (df_od
    .join(df_ord, "OrderID", "inner")
    .join(df_prod, "ProductID", "left")
    .select(
        "OrderID", "LineNo", "ProductID", "CustomerID", "EmployeeID",
        "OrderDate", "FiscalYear", "OrderYear", "OrderMonth", "OrderQuarter", "OrderSeason",
        "Quantity", "UnitPrice", "Discount", "DiscountBand",
        col("LineTotal").alias("SalesAmount"),
        spark_round(col("LineTotal") - (col("Quantity") * col("UnitCost")), 2).alias("GrossMargin"),
        spark_round(
            when(col("LineTotal") > 0,
                 (col("LineTotal") - col("Quantity") * col("UnitCost")) / col("LineTotal") * 100)
            .otherwise(lit(0.0)), 1
        ).alias("GrossMarginPct"),
        date_format(col("OrderDate"), "yyyyMMdd").cast("integer").alias("OrderDateKey"),
    )
)

df_items.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("gold_fact_sales_items")
print(f"gold_fact_sales_items: {df_items.count()} rows")
print(f"  Total SalesAmount: {df_items.agg(spark_round(spark_sum('SalesAmount'),2).alias('v')).collect()[0]['v']}")


In [ ]:
# Order-level fact. Grain: one row per order.
# Freight and TotalOrderValue belong here because they describe the shipment event,
# not the individual product. Separating grains prevents double-counting in Power BI.
# DeliveryDays is negative -- shipment dates (2007-2012) predate orders (2016-2021).
# Known legacy data issue. Flagged in ShipmentDataWarning for the end user.

from pyspark.sql.functions import (
    col, coalesce, lit, datediff, date_format,
    round as spark_round, sum as spark_sum
)

df_ord = spark.table("silver_orders")

df_ship = (spark.table("silver_shipments")
    .select("OrderID", "ShipperID", "ShipmentDate", "_data_warning")
    .dropDuplicates(["OrderID"]))

df_od_agg = (spark.table("silver_order_details")
    .groupBy("OrderID")
    .agg(spark_round(spark_sum("LineTotal"), 2).alias("TotalProductSales")))

df_orders = (df_ord.drop("ShipperID")
    .join(df_ship, "OrderID", "left")
    .join(df_od_agg, "OrderID", "left")
    .select(
        "OrderID", "CustomerID", "EmployeeID", "ShipperID",
        "OrderDate", "ShipmentDate",
        "FiscalYear", "OrderYear", "OrderMonth", "OrderQuarter", "OrderSeason",
        "Freight",
        col("TotalProductSales"),
        spark_round(col("TotalProductSales") + coalesce(col("Freight"), lit(0.0)), 2).alias("TotalOrderValue"),
        datediff(col("ShipmentDate"), col("OrderDate")).alias("DeliveryDays"),
        date_format(col("OrderDate"), "yyyyMMdd").cast("integer").alias("OrderDateKey"),
        date_format(col("ShipmentDate"), "yyyyMMdd").cast("integer").alias("ShipDateKey"),
        col("_data_warning").alias("ShipmentDataWarning"),
    ))

df_orders.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("gold_fact_orders")

total_orders = df_orders.count()
null_dates = df_orders.filter(col("OrderDate").isNull()).count()
total_val = df_orders.agg(spark_round(spark_sum("TotalOrderValue"), 2).alias("v")).collect()[0]["v"]

print(f"gold_fact_orders: {total_orders} rows")
print(f"  Null OrderDate:   {null_dates}")
print(f"  Total OrderValue: {total_val}")


In [ ]:
# gold_fact_budget
# Built from silver_budget which is already in long format (unpivoted in NB_02).
# BudgetMonthly (annual / 12) was also computed in Silver, enabling month-level
# comparison against sales in Power BI without any additional DAX measures.

from pyspark.sql.functions import col, concat_ws, round as spark_round, sum as spark_sum

df_bud = spark.table("silver_budget")
df_emp = spark.table("silver_employees").select(
    "EmployeeID",
    concat_ws(" ", col("First_Name"), col("Last_Name")).alias("EmployeeName"),
    col("Office").alias("EmployeeOffice"),
    "Title"
)

df_gold = (df_bud
    .join(df_emp, "EmployeeID", "left")
    .withColumn("BudgetYear",    col("BudgetYear").cast("integer"))
    .withColumn("BudgetAmount",  col("BudgetAmount").cast("double"))
    .withColumn("BudgetMonthly", col("BudgetMonthly").cast("double"))
)

df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("gold_fact_budget")
print(f"gold_fact_budget: {df_gold.count()} rows")
df_gold.show(10)

In [ ]:
#Data testing and validation

from pyspark.sql.functions import col, avg, sum as spark_sum, round as spark_round

print("=" * 60)
print("GOLD LAYER -- VALIDATION SUMMARY")
print("=" * 60)

# 1. Deleta a tabela antiga que foi descontinuada (evita confusão no BI)
spark.sql("DROP TABLE IF EXISTS gold_fact_sales")

# 2. Loop de validação atualizado (vai listar apenas as tabelas Gold ativas)
gold_tables = [t.tableName for t in spark.sql("SHOW TABLES").filter("tableName LIKE 'gold_%'").collect()]
for t in sorted(gold_tables):
    df = spark.table(t)
    nulls = [(c, df.filter(col(c).isNull()).count()) for c in df.columns if df.filter(col(c).isNull()).count() > 0]
    print(f"\n{t}: {df.count()} rows | {len(df.columns)} cols")
    for c, n in nulls:
        print(f"  NULL {c}: {n}")

print("\n--- Key KPIs ---")

# 3. Busca os KPIs das tabelas novas e corretas
fi = spark.table("gold_fact_sales_items")
fo = spark.table("gold_fact_orders")

total_sales  = fi.agg(spark_round(spark_sum("SalesAmount"), 2).alias("v")).collect()[0]["v"]
total_margin = fi.agg(spark_round(spark_sum("GrossMargin"), 2).alias("v")).collect()[0]["v"]
avg_margin_pct = fi.agg(spark_round(avg("GrossMarginPct"), 1).alias("v")).collect()[0]["v"]
distinct_orders = fo.select("OrderID").distinct().count()

print(f"Total SalesAmount:  {total_sales}")
print(f"Total GrossMargin:  {total_margin}")
print(f"Avg GrossMarginPct: {avg_margin_pct}%")
print(f"Distinct Orders:    {distinct_orders}")